# 🎓 LSI Design Contest 2026 - VNU-UET
## Knowledge Distillation & LFW Benchmark for TinyMobileFaceNet-32
**Tác giả:** Nguyễn Sỹ Tuấn Anh
**Mục tiêu phần cứng:** AMD Spartan-7 XC7S25CSGA225-1 (ZERO7 Board)

---
### 📌 Mục đích của Notebook này:
1. **Giải quyết bài toán thiếu dữ liệu:** Tự động tải tập dữ liệu quốc tế **LFW (13,233 ảnh, 5,749 người)**.
2. **Chuyển giao tri thức (Knowledge Distillation):** Dùng Teacher `MobileFaceNet` (112x112, 99.48% LFW) dạy cho Student phần cứng `TinyMobileFaceNet-32` (32x32, 217k params).
3. **Kiểm thử khoa học:** Đánh giá trên 6,000 cặp kiểm thử LFW (`pairs.txt`), xuất **Accuracy**, **ROC Curve**, **AUC**, **EER** cho bài báo báo cáo cuộc thi.
4. **Tải trọng số về máy:** Xuất file `tiny_mobilefacenet_lfw.pt` để nhúng vào demo camera và nạp Bitstream lên FPGA.

### Bước 1: Kiểm tra phần cứng GPU Colab (T4 / V100 / A100)

In [ ]:
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

### Bước 2: Tải tập dữ liệu chuẩn LFW và danh sách 6,000 cặp kiểm thử (pairs.txt)

In [ ]:
import os, urllib.request, tarfile
os.makedirs("lfw_dataset", exist_ok=True)

# 1. Tải pairs.txt
pairs_url = "https://huggingface.co/datasets/DerrickUnleashed/LFW/resolve/main/pairs.txt"
pairs_path = "lfw_dataset/pairs.txt"
if not os.path.exists(pairs_path):
    print("Đang tải pairs.txt...")
    !curl -L -o lfw_dataset/pairs.txt "https://huggingface.co/datasets/DerrickUnleashed/LFW/resolve/main/pairs.txt"
    print("Tải pairs.txt thành công!")

# 2. Tải lfw-funneled.tgz (~230 MB)
archive_path = "lfw_dataset/lfw-funneled.tgz"
lfw_dir = "lfw_dataset/lfw_funneled"
if not os.path.exists(lfw_dir):
    if not os.path.exists(archive_path):
        print("Đang tải lfw-funneled.tgz (~230 MB) từ mirror Hugging Face tốc độ cao...")
        !curl -L -o lfw_dataset/lfw-funneled.tgz "https://huggingface.co/datasets/DerrickUnleashed/LFW/resolve/main/lfwfunneled.tgz"
    print("Đang giải nén tập LFW...")
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(path="lfw_dataset")
    if not os.path.exists(lfw_dir) and os.path.exists("lfw_dataset/lfwfunneled"):
        os.rename("lfw_dataset/lfwfunneled", lfw_dir)
    print("Giải nén hoàn tất! Thư mục sẵn sàng:", lfw_dir)

### Bước 3: Nạp Teacher Model (MobileFaceNet 112x112)
*(Kéo thả file `mobilefacenet_scripted.pt` từ máy tính vào khung Files của Google Colab bên trái)*

In [ ]:
from google.colab import files
import os
teacher_path = "mobilefacenet_scripted.pt"
if not os.path.exists(teacher_path):
    print("Vui lòng tải file mobilefacenet_scripted.pt (7.6 MB) từ máy tính lên Colab:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if "mobilefacenet" in fn.lower():
            teacher_path = fn
            break
print("Đã xác nhận Teacher Path:", teacher_path)

### Bước 4: Tải mã nguồn huấn luyện `train_lfw_distill.py`
*(Kéo thả file `train_lfw_distill.py` từ thư mục CNN/Python vào Google Colab)*

In [ ]:
if not os.path.exists("train_lfw_distill.py"):
    print("Vui lòng tải file train_lfw_distill.py từ máy tính lên Colab:")
    files.upload()
print("Đã sẵn sàng mã nguồn train!")

### Bước 5: Khởi chạy Huấn luyện Chuyển giao Tri thức & Đánh giá LFW tự động
*(Trên GPU T4 Colab, 25 epochs mất khoảng 10-15 phút)*

In [ ]:
!python train_lfw_distill.py --epochs 25 --batch-size 64 --teacher mobilefacenet_scripted.pt --data-root ./lfw_dataset

### Bước 6: Xem Đồ thị Báo cáo Khoa học (ROC & Phân phối Cosine)

In [ ]:
from IPython.display import Image, display
display(Image("lfw_benchmark_report.png"))

### Bước 7: Tải Trọng số Đã Train Về Máy Tính Cá nhân

In [ ]:
from google.colab import files
files.download("weights/tiny_mobilefacenet_lfw.pt")
files.download("lfw_benchmark_report.png")